In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType
import time

In [2]:
spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:34:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)

In [ ]:
df.take(25)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [7]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [8]:
df_repartitioned = df.repartition(10)

In [9]:
print("After Partitions:", df_repartitioned.rdd.getNumPartitions())

[Stage 3:>                                                          (0 + 2) / 2]

After Partitions: 10


In [32]:
df_coalesced = df_repartitioned.coalesce(2)

In [33]:
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())

[Stage 10:=============================>                            (1 + 1) / 2]

After Coalesced: 2


In [ ]:
#Optimization and Caching

In [4]:
optimized_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id", "value")

In [44]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#73L, (id#73L % 1000) AS value#75L]
+- *(1) Filter (((id#73L % 1000) > 500) AND (id#73L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [5]:
start_time = time.time()
count_uncached = optimized_df.count()  # Action triggers DAG
end_time = time.time()
print(f"1. Optimized Execution  | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

[Stage 0:>                                                          (0 + 2) / 2]

1. Optimized Execution  | Count: 2495000 | Time: 2.4312 seconds


In [6]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [7]:
start_time = time.time()
count_first_cache = optimized_df.count() # Action triggers caching
end_time = time.time()
print(f"2. FIRST CACHED Action | Count: {count_first_cache} | Time: {round(end_time - start_time, 4)} seconds (Materializing cache)")

[Stage 3:=============================>                             (1 + 1) / 2]

2. FIRST CACHED Action | Count: 2495000 | Time: 2.5041 seconds (Materializing cache)


In [8]:
start_time = time.time()
count_second_cache = optimized_df.count()
end_time = time.time()
print(f"3. SECOND CACHED Action| Count: {count_second_cache} | Time: {round(end_time - start_time, 4)} seconds (Read from memory)")

3. SECOND CACHED Action| Count: 2495000 | Time: 0.1941 seconds (Read from memory)


In [56]:
optimized_df.unpersist()
print("\n--> Unpersisted DataFrame to free up memory.")


--> Unpersisted DataFrame to free up memory.


In [ ]:
#Udfs

In [3]:
data = [("Alice", 25), ("Bob", 17), ("Charlie", 35), ("David", 15)]
df1 = spark.createDataFrame(data, ["Name", "Age"])

26/06/13 06:34:16 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
df1.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [6]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [7]:
age_category_udf = udf(categorize_age, StringType())

In [8]:
df_method1 = df1.withColumn("Category", age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [ ]:
#for sql

In [9]:
spark.udf.register("sql_categorize_age", categorize_age, StringType())

df1.createOrReplaceTempView("people")

In [65]:
sql_df = spark.sql("""
    SELECT 
        Name, 
        Age, 
        sql_categorize_age(Age) AS Category 
    FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [ ]:
spark.stop()